# 03 — Image Models: CNN, ResNet-18


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from src.data.loaders import ImageFolderDataset
from src.models.image_cnn import build_image_model
from src.utils import set_seed, ensure_dir, load_config
from scripts.prepare_image_dataset import prefer_lmvd_or_synthetic

set_seed(42)
config = load_config(ROOT / "config.yaml")
device = "cpu"
test_size = float(config.get("evaluation", {}).get("test_size", 0.2))
print("Device:", device, "test_size:", test_size)


In [ ]:
image_dir = ROOT / config["paths"]["image_dir"]
lmvd_root = ROOT / "datasets" / "lmvd_extracted"
prefer_lmvd_or_synthetic(image_dir, lmvd_root, per_class=150)

transform = transforms.Compose([
    transforms.Resize((config["image_models"]["image_size"],) * 2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
dataset = ImageFolderDataset(image_dir, transform=transform)
print(f"Images: {len(dataset)}, classes: {dataset.class_to_idx}")

labels = [lab for _, lab in dataset.samples]
idx = np.arange(len(dataset))
train_idx, val_idx = train_test_split(idx, test_size=test_size, stratify=labels, random_state=42)
train_loader = DataLoader(Subset(dataset, train_idx.tolist()), batch_size=16, shuffle=True)
val_loader = DataLoader(Subset(dataset, val_idx.tolist()), batch_size=16)


In [ ]:
EPOCHS = 3
results = []
for arch in ["cnn", "resnet18"]:
    model = build_image_model(arch, num_classes=dataset.num_classes, pretrained=(arch != "cnn")).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    cw = compute_class_weight("balanced", classes=np.unique(labels), y=np.array(labels)[train_idx])
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32))
    for _ in range(EPOCHS):
        model.train()
        for images, batch_labels in train_loader:
            optimizer.zero_grad()
            criterion(model(images), batch_labels).backward()
            optimizer.step()
    model.eval()
    preds, y_true, probs = [], [], []
    with torch.no_grad():
        for images, batch_labels in val_loader:
            logits = model(images)
            p = torch.softmax(logits, dim=-1)
            preds.extend(p.argmax(1).numpy())
            y_true.extend(batch_labels.numpy())
            probs.extend(p[:, 1].numpy() if p.shape[1] == 2 else p.max(1).values.numpy())
    y_true, preds, probs = np.array(y_true), np.array(preds), np.array(probs)
    metrics = {
        "model": arch,
        "accuracy": float(accuracy_score(y_true, preds)),
        "f1_macro": float(f1_score(y_true, preds, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, preds, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    results.append(metrics)
    print(arch, metrics)

pd.DataFrame(results).to_csv(ensure_dir(ROOT / "outputs" / "results") / "image_model_results.csv", index=False)
print("03_image_models.py complete.")
